# Exercise 00 : Tokens, Tools and a Loop

- A model reads tokens and predicts the next one
- A model call keeps nothing ; the application resends the history
- A tool call is a request, and the application runs it
- An agent is a model calling tools in a loop, and every call bills the whole context

You will count tokens, read next-token probabilities, give a model a memory, run a tool
call by hand and write an agent loop. The Aurelia records are made up.

Worked examples to revisit : module 00, demo01 · tokens, demo02 · next-token, demo03 · no-memory,
demo04 · tool-call-by-hand, demo05 · agent-loop-by-hand.

Run the next cell first. Its code is folded : click the cell and press Shift+Enter. It loads the course key and builds `llm`, the course model.

In [1]:
#@title Course setup : run this cell first
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## Step 1 : Tokens

A model reads tokens, and prices and size limits are counted in tokens, as the slide *Tokens*
shows. This step cuts a service note into tokens and makes no model call.

The cell has the tokenizer in `encoding` and the note in `text`, and the `for` loop at the end
shows how five separate words are cut. Fill the first blank with the method that turns text into
token ids, and the second with the method that turns one id back into text.

The token count should be higher than the word count, and the check confirms that the pieces
rebuild the note. A `TypeError` usually means the two methods are swapped. When you are stuck,
open the collapsed Hint under the cell first, then the Solution.

In [2]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")
text = "Device CT400-2021-0117 reports error E-CT-110 and the gantry fan is loud."
ids = encoding.encode(text)
pieces = [encoding.decode([i]) for i in ids]

print("words:", len(text.split()))
print("tokens:", len(ids))
print(pieces)

for word in [" the", " scanner", " gantry", " CT400-2021-0117", " koelventilator"]:
    print(f"{word!r:20} {[encoding.decode([i]) for i in encoding.encode(word)]}")

words:

 11
tokens: 24
['Device', ' CT', '400', '-', '202', '1', '-', '011', '7', ' reports', ' error', ' E', '-', 'CT', '-', '110', ' and', ' the', ' gan', 'try', ' fan', ' is', ' loud', '.']
' the'               [' the']
' scanner'           [' scanner']
' gantry'            [' gan', 'try']
' CT400-2021-0117'   [' CT', '400', '-', '202', '1', '-', '011', '7']
' koelventilator'    [' koel', 'vent', 'ilator']


In [3]:
print("check :", "pass" if "".join(pieces) == text and len(ids) > len(text.split()) else "fail")

check : pass


## Step 2 : The next token

A model draws each token from a list of probabilities, which is why one prompt can give
different replies, as on the slides *The Next Token* and *Sampling and Temperature*.

The first half of the cell prints the likeliest tokens at each position, and `max_tokens=8`
keeps replies so short that one may stop in the middle of a sentence. The second half sends the
prompt five times through `sampler`.

The first blank turns `logprobs` on. The second is the `math` function that turns a logprob, a
logarithm, back into a probability. The third is the value that means no seed, because
`make_llm` sets a fixed seed by default.

Each `picked` line should show percentages between 0 and 100, and the count of different replies
changes from run to run. An error on the line with `reply.response_metadata["logprobs"]` means
the reply came without logprobs, so look at the first blank.

In [4]:
import math

prompt = "Complete in a few words : The CT scanner stopped because the"
picker = make_llm(max_tokens=8).bind(logprobs=True, top_logprobs=5)
reply = picker.invoke(prompt)

print("reply:", reply.content)
percents = []
for position in reply.response_metadata["logprobs"]["content"]:
    options = [(o["token"], 100 * math.exp(o["logprob"])) for o in position["top_logprobs"][:4]]
    percents += [p for _, p in options]
    print(f"picked {position['token']!r:12}", " ".join(f"{t!r} {p:.0f}%" for t, p in options))

sampler = make_llm(max_tokens=8, seed=None)
samples = [sampler.invoke(prompt).content for _ in range(5)]
print(samples)

reply: X-ray tube overheated.
picked 'X'          'power' 44% 'X' 32% 'The' 15% 'tube' 6%
picked '-ray'       '-ray' 100%
picked ' tube'      ' tube' 100%
picked ' overhe'    ' overhe' 98%
picked 'ated'       'ated' 100%
picked '.'          '.' 100%


['power went out.', 'power went out.', 'power went out.', 'power went out.', 'X-ray tube overheated.']


In [5]:
print("different replies :", len(set(samples)), "of", len(samples))
print("check :", "pass" if percents and all(0 <= p <= 100 for p in percents) and len(samples) == 5 else "fail")

different replies : 2 of 5
check : pass


## Step 3 : No memory

Each model call starts from nothing, and a chat application creates its memory by resending the
conversation, as the slide *No Memory Between Calls* shows.

The first lines of the cell show the problem : in two separate calls, the second has never seen
the serial number. The `for` loop keeps `history`, a list of `("human", text)` and `("ai",
text)` pairs. Fill the first blank with what the model receives on each call, and the second
with the text of its reply, so the next pass includes it.

The `input tokens` column should rise on every call, and the last reply should name the serial
number. Input tokens that stay about the same mean the model receives only the newest line, and
a `ValidationError` means the whole reply object went into the list in place of its text.

In [6]:
SERIAL = "CT400-2021-0117"

llm.invoke(f"My device is {SERIAL}. Reply with one word : noted.")
forgot = llm.invoke("Which serial number did I give you? Reply in one sentence.")
print("no history :", forgot.content)

history = []
for line in [f"My device is {SERIAL}. Reply with one word : noted.",
             "Its gantry is overheating. Reply with one word : noted.",
             "Which serial number did I give you, and what is wrong with it? Reply in one sentence."]:
    history.append(("human", line))
    reply = llm.invoke(history)
    history.append(("ai", reply.content))
    print(f"input tokens {reply.usage_metadata['input_tokens']:3}  {reply.content}")

no history : You haven’t given me a serial number in this conversation.


input tokens  25  noted


input tokens  50  noted


input tokens  82  The serial number is CT400-2021-0117, and its gantry is overheating.


In [7]:
remembered = SERIAL in history[-1][1]
print("check :", "pass" if SERIAL not in forgot.content and remembered else "fail")

check : pass


## Step 4 : A tool call by hand

A tool call is a request : the model asks, and your code runs the tool and sends back the
result, as on the slide *What is Tool Calling?*.

The first cell defines three tools that read Aurelia's `service.db`, the list `tools`, and
`by_name`, a dictionary from tool name to tool. In the second cell `call` is the model's first
request, a dictionary with the keys `name`, `args` and `id`.

Fill the first blank so `by_name` finds the tool the model named. Fill the two `ToolMessage`
blanks with the tool's result and the id of the request, because the id ties the result to the
request.

The run worked when the `our code ran` line shows the device's record, and the check tests that
record and the id. A `KeyError` on `by_name` means the lookup key is wrong. When the model answers
in text and asks for no tool, the cell stops with a message that says so : run the cell again.
`check : fail` with another tool on the `tool ran` line means the model chose a tool that does
not return the device, so run the cell again as well.

In [8]:
import json
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = Path(data("capstone/service.db")).resolve()


def rows(sql, *args):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        return [dict(row) for row in db.execute(sql, args)]


@tool
def work_order(wo_id: str) -> str:
    """Look up an Aurelia work order by id, such as WO-9001. Returns the serial, status, error code and summary."""
    found = rows("SELECT wo_id, serial, status, error_code, summary FROM work_orders WHERE wo_id = ?", wo_id.strip().upper())
    return json.dumps(found[0]) if found else "Unknown work order"


@tool
def device_info(serial: str) -> str:
    """Look up an Aurelia device by serial number. Returns its model, site and firmware."""
    found = rows("SELECT serial, model, site_id, firmware FROM devices WHERE serial = ?", serial.strip().upper())
    return json.dumps(found[0]) if found else "Unknown device"


@tool
def parts_for_model(model: str) -> str:
    """List the spare parts of a device model, such as CT-400. Returns part number, name, stock and depot."""
    return json.dumps(rows("SELECT part_no, name, stock, depot FROM parts WHERE model = ?", model.strip().upper()))


tools = [work_order, device_info, parts_for_model]
by_name = {t.name: t for t in tools}
print([t.name for t in tools])

['work_order', 'device_info', 'parts_for_model']


In [9]:
from langchain.messages import ToolMessage

model = llm.bind_tools(tools)
messages = [("human", f"Which model is device {SERIAL}? Use a tool.")]
request = model.invoke(messages)
print("text:", repr(request.content))
print("tool calls:", request.tool_calls)

if not request.tool_calls:
    raise RuntimeError("The model replied in text and asked for no tool, so there is nothing to run. Run this cell again.")
call = request.tool_calls[0]
result = by_name[call["name"]].invoke(call["args"])
print("our code ran", call["name"], "->", result)

messages += [request, ToolMessage(result, tool_call_id=call["id"])]
final = model.invoke(messages)
print("text:", final.content)
print("tool calls:", final.tool_calls)

text: ''
tool calls: [{'name': 'device_info', 'args': {'serial': 'CT400-2021-0117'}, 'id': 'call_zotS2y8wDbBv3DoRPdEBmaTS', 'type': 'tool_call'}]
our code ran device_info -> {"serial": "CT400-2021-0117", "model": "CT-400", "site_id": "S01", "firmware": "5.3"}


text: Device CT400-2021-0117 is a **CT-400** model.
tool calls: []


In [10]:
returned = SERIAL in result and "CT-400" in result
tied = messages[-1].content == result and messages[-1].tool_call_id == call["id"]
print("tool ran :", call["name"])
print("returned the device :", returned)
print("result tied to the request :", tied)
print("check :", "pass" if returned and tied else "fail")

tool ran : device_info
returned the device : True
result tied to the request : True
check : pass


## Step 5 : An agent loop

An agent is the round of Step 4 repeated until the model stops asking for tools, and every call
bills the whole context again, as on the slides *The Agent Loop*, *How an Agent Stops* and
*Tokens Across a Run*.

The cell uses `model`, `by_name` and `ToolMessage` from Step 4. The `while` loop is given :
`MAX_CALLS` is the cap, `calls` records the input and output tokens of each call, and the inner
`for` runs the tools the model asked for.

The first blank sends the model every message so far. The second is what the `if not` tests, so
a reply without tool calls ends the loop. The third adds up the input tokens : `calls` is a list
of pairs, and `for i, _ in calls` gives each input count the name `i`.

The `input` column should grow, with no tools on the last row. `loop ended itself : False` means
`MAX_CALLS` stopped the run, often because the second blank is wrong.

The check grades your loop on its structure : every tool call got a result, the last reply asks
for no tool, and the total billed exceeds the last context. The model's wording plays no part in
it. `answer matches service.db` is a separate fact : `False` does not fail the check, and it asks
you to read the reply against the `database says` line.

In [11]:
from langchain.messages import HumanMessage, SystemMessage

MAX_CALLS = 6
messages = [
    SystemMessage("You are the service desk assistant of Aurelia Imaging Systems. Use the tools. Answer in two sentences."),
    HumanMessage("Work order WO-9001 reports a problem. Look up the work order, then the device, then the parts "
                 "of its model. Which part is the likely replacement, how many are in stock, and at which depot?"),
]

calls = []
while len(calls) < MAX_CALLS:
    reply = model.invoke(messages)
    messages.append(reply)
    usage = reply.usage_metadata
    calls.append((usage["input_tokens"], usage["output_tokens"]))
    print(f"call {len(calls)}  input {usage['input_tokens']:5}  output {usage['output_tokens']:4}  "
          f"tools {[c['name'] for c in reply.tool_calls]}")
    if not reply.tool_calls:
        break
    for call in reply.tool_calls:
        result = by_name[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(result, tool_call_id=call["id"]))

print()
print(reply.content)

billed = sum(i for i, _ in calls)
written = sum(o for _, o in calls)
print("model calls:", len(calls))
print("context of the last call:", calls[-1][0], "tokens")
print("input tokens billed in total:", billed)
# gpt-6-luna on OpenRouter : $0.10 per million input tokens, $0.50 per million output tokens
print(f"cost at list price: ${(billed * 0.10 + written * 0.50) / 1_000_000:.6f}")

call 1  input   207  output   22  tools ['work_order']


call 2  input   292  output   25  tools ['device_info']


call 3  input   366  output   21  tools ['parts_for_model']


call 4  input   537  output   41  tools []

The likely replacement is the gantry cooling fan (part CT-FN-02), given the gantry over-temperature report. There are 6 in stock at the Houten depot.
model calls: 4
context of the last call: 537 tokens
input tokens billed in total: 1402
cost at list price: $0.000195


In [12]:
fan = rows("SELECT part_no, stock, depot FROM parts WHERE part_no = 'CT-FN-02'")[0]
matches = all(str(value) in reply.content or value == "CT-FN-02" and "fan" in reply.content.lower()
              for value in fan.values())
asked = [c["id"] for m in messages if m.type == "ai" for c in m.tool_calls]
answered = [m.tool_call_id for m in messages if m.type == "tool"]
tools_ran = bool(asked) and asked == answered
print("database says :", fan)
print("answer matches service.db :", matches)
print("tools ran :", tools_ran)
print("loop ended itself :", not reply.tool_calls)
print("billed more than the last context :", billed > calls[-1][0])
print("check :", "pass" if tools_ran and not reply.tool_calls and billed > calls[-1][0] else "fail")

database says : {'part_no': 'CT-FN-02', 'stock': 6, 'depot': 'Houten'}
answer matches service.db : True
tools ran : True
loop ended itself : True
billed more than the last context : True
check : pass
